In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [2]:
load_dotenv("local.env")

DB_USER = "postgres"
DB_PASSWORD = os.getenv("DB_PW")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "airline_data_new"

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [15]:
with engine.connect() as conn:
    profiles = pd.DataFrame(conn.execute(text("""

with booking_stats as (
    select
        coalesce(traveller_type_id, 'ALL_TYPES') as traveller_type_id,
        100.0 * count(*) / (select count(*) from bookings) as booking_share_pct,
        avg(bookings_per_customer) / ((max(flight_date) - min(flight_date)) / 365.25)
            as avg_yrly_bkgs_per_cust,
        count(case when class_id != '(01) Economy' then 1 end) * 100.0 / count(*) as premium_cabin_usage_pct,
        100.0 * count(case when is_checked_in = TRUE and has_cancellation_refund = FALSE then 1 end)
            / count(*) filter (where has_cancellation_refund = FALSE) as avg_check_in_rate_pct,
        avg(flight_date::date - booking_time::date)::int as avg_bkg_lead_time_days,
        avg(price_paid / expd_avg_price) as avg_final_to_base_price_ratio
    from bookings_view
    group by cube (traveller_type_id)
),

status_counts as (
    select
        coalesce(traveller_type_id, 'ALL_TYPES') as traveller_type_id,
        frequent_flyer_status_id,
        count(*) as cnt
    from customers
    group by grouping sets ((traveller_type_id, frequent_flyer_status_id), (frequent_flyer_status_id))
),

status_dist as (
    select
        *,
        row_number() over (partition by traveller_type_id order by cnt desc) as rn
    from status_counts
),

status_summary as (
    select
        traveller_type_id,
        max(frequent_flyer_status_id) filter (where rn = 1) as dominant_loyalty_status,
        max(frequent_flyer_status_id) filter (where rn = 2) as secondary_loyalty_status
    from status_dist
    group by 1
),

customer_stats as (
    select
        coalesce(c.traveller_type_id, 'ALL_TYPES') as traveller_type_id,
        100.0 * count(*) / (select count(*) from customers) as customer_share_pct,
        100.0 * count(*) filter (where gender = 'Female') / count(*) as female_ratio_pct,
        percentile_cont(0.5) within group (order by c.age_canonical) as median_age
    from customers_view c
    group by cube (c.traveller_type_id)
)

select
    cs.traveller_type_id,
    ss.dominant_loyalty_status,
    ss.secondary_loyalty_status,
    cs.customer_share_pct,
    cs.female_ratio_pct,
    bs.booking_share_pct,
    bs.avg_yrly_bkgs_per_cust,
    cs.median_age,
    bs.premium_cabin_usage_pct,
    bs.avg_check_in_rate_pct,
    bs.avg_bkg_lead_time_days,
    bs.avg_final_to_base_price_ratio
from customer_stats cs
join booking_stats bs using (traveller_type_id)
join status_summary ss using (traveller_type_id)
order by 1;
""")))

profiles

,traveller_type_id,dominant_loyalty_status,secondary_loyalty_status,customer_share_pct,female_ratio_pct,booking_share_pct,avg_yrly_bkgs_per_cust,median_age,premium_cabin_usage_pct,avg_check_in_rate_pct,avg_bkg_lead_time_days,avg_final_to_base_price_ratio
0,(01) Leisure,(0) No Status,None,53.9711784773177546,49.0199859349833261,18.9175164915519730,1.2025197106513106,43.0,1.5314807329542839,89.8741001302002518,40,1.22758996915028986894
1,(02) Family,(0) No Status,None,15.8414984035657100,50.6406347952287307,5.5415430352952914,1.1956413414495155,43.0,0.67975549513688220872,89.6673328346904442,49,1.14590116979418443682
2,(03) Corporate,(1) Silver,(2) Gold,23.5427010461961512,45.2361461563649672,39.9506402436706938,5.4340480099568066,46.0,13.1205266256927221,92.2174670420205241,13,1.38185618445958579200
3,(04) Road warrior,(2) Gold,(3) Platinum,6.6446220729203842,42.9065236233100615,35.5903002294820417,16.8266937189756644,49.0,22.6234972929561982,93.3846529771156672,8,1.40523823910443876746
4,ALL_TYPES,(0) No Status,(1) Silver,100.0000000000000000,47.9796864449782267,100.0000000000000000,8.4533516284658582,44.0,13.6208920627967411,92.0480354317247687,18,1.34791904250386085376


In [11]:
with engine.connect() as conn:
    age_groups = pd.DataFrame(conn.execute(text("""

with booking_stats as (
    select
        traveller_type_id,
        age_group,
        100.0 * count(*) / (select count(*) from bookings) as booking_share_pct,
        count(case when class_id != '(01) Economy' then 1 end) * 100.0 / count(*) as premium_cabin_usage_pct,
        100.0 * count(case when is_checked_in = TRUE and has_cancellation_refund = FALSE then 1 end)
            / count(*) filter (where has_cancellation_refund = FALSE) as avg_check_in_rate_pct,
        avg(flight_date::date - booking_time::date)::int as avg_bkg_lead_time_days,
        avg(price_paid / expd_avg_price) as avg_final_to_base_price_ratio
    from bookings_view
    group by traveller_type_id, age_group
),

customer_stats as (
    select
        traveller_type_id,
        age_group,
        100.0 * count(*) / (select count(*) from customers) as customer_share_pct,
        100.0 * count(*) filter (where gender = 'Female') / count(*) as female_ratio_pct,
        100.0 * count(*) filter (where gender = 'Male') / count(*) as male_ratio_pct
    from customers_view
    group by traveller_type_id, age_group
)

select
    cs.traveller_type_id,
    cs.age_group,
    cs.customer_share_pct,
    cs.female_ratio_pct,
    cs.male_ratio_pct,
    bs.booking_share_pct,
    bs.premium_cabin_usage_pct,
    bs.avg_check_in_rate_pct,
    bs.avg_bkg_lead_time_days,
    bs.avg_final_to_base_price_ratio
from customer_stats cs
join booking_stats bs
using (traveller_type_id, age_group)
order by 1, 2;
""")))

age_groups

,traveller_type_id,age_group,customer_share_pct,female_ratio_pct,male_ratio_pct,booking_share_pct,premium_cabin_usage_pct,avg_check_in_rate_pct,avg_bkg_lead_time_days,avg_final_to_base_price_ratio
0,(01) Leisure,Age 18–24,4.7362697808434234,48.8398255549544978,51.1601744450455022,1.6603204674150552,1.5386924297848408,88.2039674513991929,32,1.26244023116410900270
1,(01) Leisure,Age 25–34,12.3251937328900834,47.8050835453470126,52.1949164546529874,4.3192228247116524,1.5257785065866686,88.7530301819441115,35,1.24838805903892525831
2,(01) Leisure,Age 35–44,11.2113628211517454,48.8397725250368069,51.1602274749631931,3.9315090718323039,1.5340569346262041,90.2460034336579632,39,1.22732953100169997136
3,(01) Leisure,Age 45–60,13.3797790431551945,49.5601660032299118,50.4398339967700882,4.6880995889740894,1.5292618160225663,91.2148199296886635,44,1.20880710045962960832
4,(01) Leisure,Age 61+,12.3185730992773080,49.8821104168008761,50.1178895831991239,4.3183645386188722,1.5344748298217463,89.8433048346789405,43,1.21401670236902010287
5,(02) Family,Age 18–24,0.61540149834914954333,53.4227396654631199,46.5772603345368801,0.21590551624247281838,0.68197715085101422243,87.9157884536182378,41,1.17435065350086492292
6,(02) Family,Age 25–34,3.6819792248144976,53.1627173752401596,46.8372826247598404,1.2881942974465208,0.66998304038264648186,88.4304987361560873,44,1.16204543103532994135
7,(02) Family,Age 35–44,4.2322627103694994,50.5239416699703207,49.4760583300296793,1.4797934535776209,0.66947086968280075588,89.8600577012432445,48,1.14722513242053546450
8,(02) Family,Age 45–60,4.6127450824790373,49.5672006409657742,50.4327993590342258,1.6134041836337578,0.69593189508448372961,90.7128049442667803,53,1.13360756797654295957
9,(02) Family,Age 61+,2.6991098875535262,48.5832850314592206,51.4167149685407794,0.94424558439491912755,0.68105727811660340342,89.6664912329298357,52,1.13630198446952460409


In [14]:
with engine.connect() as conn:
    classes = pd.DataFrame(conn.execute(text("""

select
    class_id as cabin_class,
    age_group,
    coalesce(gender, 'OVERALL') as gender,
    case
        when grouping(class_id) = 1 then 100.0
        else 100.0 * count(*) / sum(count(*)) over (partition by age_group, gender)
    end as booking_share_pct,
    100.0 * count(*) filter (where is_checked_in and has_cancellation_refund = false)
        / count(*) filter (where has_cancellation_refund = false) as avg_check_in_rate_pct
from bookings b
join customers_view c using (customer_id)
group by grouping sets (
   (class_id, age_group, gender),
   (class_id, age_group)
)
order by 1, 2, 3;
""")))

classes

,cabin_class,age_group,gender,booking_share_pct,avg_check_in_rate_pct
0,(01) Economy,Age 18–24,Female,90.1139059753954306,89.3955097614157375
1,(01) Economy,Age 18–24,Male,88.0687551111346335,89.6175094509867898
2,(01) Economy,Age 18–24,OVERALL,88.9631013006625799,89.5191341959297249
3,(01) Economy,Age 25–34,Female,88.6534304724426691,90.2055250902901526
4,(01) Economy,Age 25–34,Male,87.6132739853354453,90.3425788135681922
5,(01) Economy,Age 25–34,OVERALL,88.0851089190340867,90.2800132780349579
6,(01) Economy,Age 35–44,Female,86.7546780270793510,92.0319229831930432
7,(01) Economy,Age 35–44,Male,86.1410797159126966,92.1079862384013420
8,(01) Economy,Age 35–44,OVERALL,86.4224286286831914,92.0729780665471619
9,(01) Economy,Age 45–60,Female,85.3949195090023168,93.0696189409959779


In [16]:
with engine.connect() as conn:
    lead_times = pd.DataFrame(conn.execute(text("""

with lead_time_bins as (
    select
        traveller_type_id,
        (flight_date::date - booking_time::date) as booking_lead_time_days,
        case
            when (flight_date::date - booking_time::date) <= 3 then '(1) <= 3 days'
            when (flight_date::date - booking_time::date) <= 7 then '(2) <= 7 days'
            when (flight_date::date - booking_time::date) <= 14 then '(3) <= 14 days'
            when (flight_date::date - booking_time::date) <= 30 then '(4) <= 30 days'
            when (flight_date::date - booking_time::date) <= 60 then '(5) <= 60 days'
            else '(6) > 60 days'
        end as lead_time_bin,
        count(*) over () as total_bookings,
        case when class_id != '(01) Economy' then 1 end as premium_cabin,
        price_paid / expd_avg_price as final_to_base_price_ratio,
        case when price_paid > expd_avg_price then 1 end as above_base_price,
        case when price_paid < expd_avg_price then 1 end as below_base_price
    from bookings_view
)

select
    traveller_type_id,
    lead_time_bin,
    100.0 * count(*) / max(total_bookings) as booking_share_pct,
    100.0 * count(premium_cabin) / count(*) as premium_cabin_usage_pct,
    avg(booking_lead_time_days)::int as avg_bkg_lead_time_days,
    avg(final_to_base_price_ratio) as avg_final_to_base_price_ratio,
    (percentile_cont(0.5) within group (order by final_to_base_price_ratio)::decimal)
        as median_final_to_base_price_ratio,
    100.0 * count(above_base_price) / count(*) as share_above_base_price_pct,
    100.0 * count(below_base_price) / count(*) as share_below_base_price_pct
from lead_time_bins
group by traveller_type_id, lead_time_bin
order by 1, 2;
""")))

lead_times

,traveller_type_id,lead_time_bin,booking_share_pct,premium_cabin_usage_pct,avg_bkg_lead_time_days,avg_final_to_base_price_ratio,median_final_to_base_price_ratio,share_above_base_price_pct,share_below_base_price_pct
0,(01) Leisure,(1) <= 3 days,0.26729193521666639567,1.3729330671588384,1,1.49141901611147392182,1.56359975947918,82.3835172699536706,17.6145994199404874
1,(01) Leisure,(2) <= 7 days,0.20022732248315405893,0.70772209023142386645,6,1.43917047602241153033,1.50459403712732,81.7035612374451610,18.2951817073324031
2,(01) Leisure,(3) <= 14 days,0.99329374008408988777,0.58002376855809993437,12,1.37100645891994685977,1.43394406943105,79.9059393521674239,20.0912732902728823
3,(01) Leisure,(4) <= 30 days,5.6025090345308379,0.71449879059938110316,23,1.30570216678593888815,1.36625691898498,77.8778972602493558,22.1182391270751194
4,(01) Leisure,(5) <= 60 days,9.0422176582320887,1.4415571979878102,43,1.18400467963296262366,1.24021328931964,71.7907620979684365,28.2032810541059940
5,(01) Leisure,(6) > 60 days,2.8119768010051361,3.8581893434347051,77,1.12131068023029813203,1.17844564635723,67.3230947146815991,32.6720718075774610
6,(02) Family,(1) <= 3 days,0.03032443063289009694,0.51460823373173970784,1,1.42831826095426809855,1.49692884843036,81.2416998671978752,18.7583001328021248
7,(02) Family,(2) <= 7 days,0.01296238527219322703,0.25242718446601941748,6,1.37815806197085365851,1.43842891334051,80.3689320388349515,19.6310679611650485
8,(02) Family,(3) <= 14 days,0.10059062668023345986,0.19266858501188539972,12,1.31557512014986703296,1.37615373521777,78.5437257600400350,21.4537720505442262
9,(02) Family,(4) <= 30 days,1.0175396929631263,0.22188031820160683896,24,1.25174760052410645521,1.309036401831,75.8434915106660862,24.1542822654054696


In [19]:
for name, df in {
    "traveller_type_profile": profiles,
    "traveller_type_age_group_breakdown": age_groups,
    "cabin_preference_by_age_and_gender": classes,
    "traveller_type_lead_time_bin_breakdown": lead_times
}.items():
    df.to_csv(f"../assets/query_output_csvs/{name}.csv", index=False)